# 02 Train

If the last run died with `No space left on device`, do **Runtime → Disconnect and delete runtime** first.

This writes a **100k English + Hindi/Marathi subset onto Google Drive in 128-clip batches**. Audio stays on disk. Peak RAM is about one batch plus the model, not 100k clips. Re-running reuses the Drive folder.

There are no separate Hinglish files.

In [ ]:
from google.colab import drive
import os, shutil, subprocess, sys
from pathlib import Path

drive.mount("/content/drive")
# Drop the failed full-dataset cache so Colab has disk again.
for cache in [Path.home() / ".cache/huggingface", Path("/root/.cache/huggingface")]:
    if cache.exists():
        shutil.rmtree(cache, ignore_errors=True)

repo = Path("/content/ShipRocket-assesment")
if repo.exists():
    subprocess.check_call(["git", "-C", str(repo), "pull"])
else:
    subprocess.check_call(
        ["git", "clone", "https://github.com/Saaalil/ShipRocket-assesment.git", str(repo)]
    )
os.chdir(repo)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".[train]"])

In [ ]:
# Builds 100k clips on Drive (disk), then trains. Reuses the subset if it already exists.
!python scripts/train.py --config configs/head_only.yaml